<a href="https://colab.research.google.com/github/YJ-Kim331/2025_Algorithm/blob/main/Assignment_2_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

## Code of MST-based Algorithm

In [3]:
import math

def euclidean(p1, p2):
    # returns euclidean distance between two 2D points
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def build_complete_graph(points):
    # builds a full distance graph between all point pairs
    n = len(points)
    graph = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                graph[i][j] = euclidean(points[i], points[j])
    return graph

def prim_mst(graph):
    # constructs MST using Prim's algorithm
    n = len(graph)
    visited = [False] * n
    parent = [-1] * n
    key = [float('inf')] * n
    key[0] = 0

    for _ in range(n):
        u = min((k for k in range(n) if not visited[k]), key=lambda x: key[x])
        visited[u] = True
        for v in range(n):
            if graph[u][v] and not visited[v] and graph[u][v] < key[v]:
                key[v] = graph[u][v]
                parent[v] = u
    return parent

def build_mst_adjacency(parent):
    # converts parent list to adjacency list for MST
    n = len(parent)
    mst = [[] for _ in range(n)]
    for v in range(1, n):
        u = parent[v]
        mst[u].append(v)
        mst[v].append(u)
    return mst

def dfs(mst, start, visited, tour):
    # performs preorder DFS traversal to build the tour
    visited[start] = True
    tour.append(start)
    for neighbor in mst[start]:
        if not visited[neighbor]:
            dfs(mst, neighbor, visited, tour)

def mst_approx_tsp(points):
    # main function: builds MST, traverses it, returns TSP tour
    graph = build_complete_graph(points)
    parent = prim_mst(graph)
    mst = build_mst_adjacency(parent)
    visited = [False] * len(points)
    tour = []
    dfs(mst, 0, visited, tour)
    tour.append(0)  # return to start
    return tour


# Code of Held-Karp Algorithm

In [4]:
import itertools

def held_karp(points):
    n = len(points)
    dist = [[euclidean(points[i], points[j]) for j in range(n)] for i in range(n)]

    C = {}  # DP table
    for k in range(1, n):
        C[(1 << k, k)] = (dist[0][k], 0)

    for subset_size in range(2, n):
        for subset in itertools.combinations(range(1, n), subset_size):
            bits = sum(1 << bit for bit in subset)
            for k in subset:
                prev = bits & ~(1 << k)
                res = []
                for m in subset:
                    if m == k:
                        continue
                    res.append((C[(prev, m)][0] + dist[m][k], m))
                C[(bits, k)] = min(res)

    bits = (1 << n) - 2  # all except starting point
    res = [(C[(bits, k)][0] + dist[k][0], k) for k in range(1, n)]
    opt, _ = min(res)
    return opt


# Test Code

In [7]:
def compute_tour_cost(tour, points):
    # calculates total length of the tour
    return sum(euclidean(points[tour[i]], points[tour[i+1]]) for i in range(len(tour)-1))

def load_tsp_coordinates(filename):
    # parses .tsp file from TSPLIB format
    points = []
    with open(filename, 'r') as f:
        reading = False
        for line in f:
            if "NODE_COORD_SECTION" in line:
                reading = True
                continue
            if "EOF" in line or not reading:
                continue
            parts = line.strip().split()
            if len(parts) >= 3:
                x, y = float(parts[1]), float(parts[2])
                points.append((x, y))
    return points

# load a280.tsp and run experiments
points = load_tsp_coordinates("a280.tsp")

# MST-based approximation
mst_tour = mst_approx_tsp(points)
print("MST-based Tour Cost:", compute_tour_cost(mst_tour, points))

# Held-Karp on first 15 cities
small_points = points[:15]
print("Held-Karp Tour Cost (15 cities):", held_karp(small_points))


MST-based Tour Cost: 3484.853377045222
Held-Karp Tour Cost (15 cities): 238.8919583350927
